# 동일 조건 반복성 보고서

같은 조건에서 생성한 여러 answers의 **평가 분포**를 비교합니다. 전체 문항과 개별 문항의 등급 건수·비율을 선택해 볼 수 있으며, 회차별 개별 판정은 원본 analysis에 보존합니다.


In [ ]:
import os
import mlflow
import pandas as pd
import panel as pn
from IPython.display import display
from elice_reporting import find_root, from_mlflow, from_files, export_html
from elice_reporting.repeatability_sections import repeatability_conditions, boundary_change_summary, repeatability_metric_guide
from elice_reporting.repeatability_view import summary_heatmaps

pn.extension()
ROOT = find_root()


In [ ]:
# 보고서마다 이 셀만 수정합니다. 실제 주소와 인증정보는 환경변수에 둡니다.
SOURCE_MODE = "mlflow"  # mlflow | local
RUN_ID = ""
REPORT_TITLE = "동일 조건 반복성"
OUTPUT_HTML = ROOT / "artifacts/reports/repeatability.html"
LOCAL_REPORT = ROOT / "artifacts/evaluation/repeatability-result.json"
LOCAL_INPUTS = ROOT / "artifacts/evaluation/repeatability-inputs.json"


In [ ]:
if SOURCE_MODE == "mlflow":
    mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
    client = mlflow.MlflowClient()
    if not RUN_ID:
        experiment = client.get_experiment_by_name(os.environ.get("QA_MLFLOW_EXPERIMENT_NAME", "elice-techblog-rag-evaluation"))
        if experiment is None:
            raise ValueError("실험을 찾을 수 없습니다. 환경 설정을 확인하세요.")
        runs = client.search_runs([experiment.experiment_id], filter_string="tags.`evaluation.kind` = 'repeatability' AND attributes.status = 'FINISHED'", max_results=20)
        display(pd.DataFrame([{"run_id": r.info.run_id, "name": r.data.tags.get("mlflow.runName")} for r in runs]))
        raise ValueError("위 목록에서 RUN_ID를 설정하고 다시 실행하세요.")
    data = from_mlflow("repeatability", RUN_ID, ROOT / "artifacts/reports/cache")
elif SOURCE_MODE == "local":
    data = from_files("repeatability", LOCAL_REPORT, LOCAL_INPUTS)
else:
    raise ValueError("SOURCE_MODE는 mlflow 또는 local이어야 합니다.")


## 1. 평가 조건

Analysis 보고서와 같은 범주별 설정을 표시합니다. Dataset·답변/검색 설정·Corpus/청킹·임베딩/인덱스·Agent·생성 환경과 원본 Run 링크를 확인할 수 있습니다.


In [ ]:
conditions_view = repeatability_conditions(data)


## 2. 지표 정의와 판정 기준

원본 analysis와 같은 rubric 등급 기준을 보여주고, 경계 충족률·시행 간 평균 변화폭의 계산 및 제외 규칙을 설명합니다.


## 3. 시행 간 평균 변화폭

질문을 대응시켜 계산한 두 경계의 평균 변화폭과 평가 주체·항목별 비교 가능 건수를 보여줍니다.


## 4. 전체 평가 비교

질문 선택에서 전체 합산 또는 개별 질문을 고릅니다. Gold의 두 rubric 경계를 별도 구역에 표시하고, 각 구역의 answers 매트릭스는 2열로 배치합니다. 각 칸은 **선택한 범위의 유효 채점 관측**에서 해당 경계를 충족한 비율입니다. 아이콘은 평가 주체를 뜻하며, hover에서 원점수·충족/유효 건수·N/A·오류·미기록을 확인할 수 있습니다.


In [ ]:
definitions_view = repeatability_metric_guide(data)
change_view = boundary_change_summary(data)
matrices = summary_heatmaps(data)


In [ ]:
report_view = pn.Column(
    pn.pane.Markdown("# " + REPORT_TITLE),
    pn.pane.Markdown("## 1. 평가 조건"), conditions_view,
    pn.pane.Markdown("## 2. 지표 정의와 판정 기준"), definitions_view,
    pn.pane.Markdown("## 3. 시행 간 평균 변화폭"), change_view,
    pn.pane.Markdown("## 4. 전체 평가 비교"), matrices,
    pn.pane.Markdown("각 비율의 분모는 유효 채점 관측입니다. 같은 답변의 Judge 반복을 각각 포함하며, 회차별 개별 판정과 사유는 원본 analysis에서 확인할 수 있습니다."),
    sizing_mode="stretch_width", max_width=1180, styles={"margin":"0 auto"},
)
report_view


## HTML export

저장된 결과만 로컬 HTML로 내보냅니다.


In [ ]:
html_path = export_html(report_view, OUTPUT_HTML, title=REPORT_TITLE, overwrite=True)
html_path
